# 09 — UDFs & Pandas UDFs

**What you will learn:**
- When to use UDFs vs built-in functions (and why built-ins should be preferred)
- Regular Python UDFs: define, register, apply
- UDF with complex return types (StructType, ArrayType)
- Pandas (Vectorized) UDFs: scalar and grouped map
- `applyInPandas` — grouped transformation using Pandas
- Performance: Python UDF vs Pandas UDF vs built-in

## When to Use UDFs

| Option | When to Use | Performance |
|---|---|---|
| Built-in `pyspark.sql.functions` | Always — covers 95% of use cases | Best (JVM, Catalyst-optimized) |
| Python UDF | Custom logic not in built-ins, simple scalar transforms | Slowest (row-by-row, serialization overhead) |
| Pandas UDF (vectorized) | Custom logic needing numpy/pandas, ML scoring | Fast (columnar, Apache Arrow) |
| `applyInPandas` | Complex grouped transformations, per-group model inference | Medium |

## Setup

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, ArrayType
)
from pyspark.sql.functions import col, udf, pandas_udf, PandasUDFType
import pyspark.sql.functions as F
import pandas as pd
import numpy as np

data = [
    (1, "Alice Smith",   "Engineering", 95000.0,  "2022-01-15"),
    (2, "Bob Jones",     "Marketing",   72000.0,  "2021-06-01"),
    (3, "Charlie Brown", "Engineering", 105000.0, "2020-03-20"),
    (4, "Diana Prince",  "HR",          68000.0,  "2023-09-10"),
    (5, "Eve Adams",     "Marketing",   78000.0,  "2022-11-05"),
    (6, "Frank Castle",  "Engineering", 88000.0,  "2021-04-12"),
]

df = spark.createDataFrame(data, ["emp_id", "name", "department", "salary", "hire_date"])
df.show()

## 1. Regular Python UDF — Scalar (Row by Row)

In [0]:
# Define a Python function
def salary_band(salary):
    if salary is None:
        return "Unknown"
    elif salary >= 100000:
        return "Band A"
    elif salary >= 80000:
        return "Band B"
    elif salary >= 70000:
        return "Band C"
    else:
        return "Band D"

# Register as a UDF with return type
salary_band_udf = udf(salary_band, StringType())

# Apply to DataFrame
df.withColumn("salary_band", salary_band_udf(col("salary"))).show()

In [0]:
# Alternative: decorator syntax
@udf(returnType=StringType())
def get_first_name(full_name):
    if full_name is None:
        return None
    parts = full_name.strip().split(" ")
    return parts[0] if parts else None

df.withColumn("first_name", get_first_name(col("name"))).show()

In [0]:
# Register UDF for use in spark.sql() string queries
spark.udf.register("salary_band_sql", salary_band, StringType())

df.createOrReplaceTempView("employees")
spark.sql("""
    SELECT emp_id, name, salary, salary_band_sql(salary) AS band
    FROM employees
""").show()

## 2. UDF Returning a StructType (Composite Return)

In [0]:
# Parse a full name into first + last name parts
name_schema = StructType([
    StructField("first", StringType(), True),
    StructField("last",  StringType(), True),
])

@udf(returnType=name_schema)
def parse_name(full_name):
    if full_name is None:
        return {"first": None, "last": None}
    parts = full_name.strip().split(" ", 1)
    return {"first": parts[0], "last": parts[1] if len(parts) > 1 else None}

df_parsed = df.withColumn("name_parts", parse_name(col("name")))
df_parsed.select(
    col("emp_id"),
    col("name_parts.first").alias("first_name"),
    col("name_parts.last").alias("last_name")
).show()

## 3. UDF Returning an ArrayType

In [0]:
# Generate salary history as a list of 3 years
@udf(returnType=ArrayType(DoubleType()))
def salary_history(current_salary, years=3):
    if current_salary is None:
        return []
    return [round(current_salary * (0.95 ** i), 2) for i in range(years)]

df.withColumn("salary_history", salary_history(col("salary"))).show(truncate=False)

## 4. Pandas UDF — Vectorized Scalar UDF

Operates on a `pandas.Series` instead of row-by-row.
Arrow serialization makes it much faster than regular UDFs.

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

# Pandas UDF: scalar — input and output are pandas Series
@pandas_udf(DoubleType())
def apply_tax(salary: pd.Series) -> pd.Series:
    # Apply a tiered tax: 30% above 90k, 22% otherwise
    return salary.apply(lambda s: s * 0.70 if s >= 90000 else s * 0.78)

df.withColumn("net_salary", apply_tax(col("salary"))).show()

In [0]:
# Pandas UDF using numpy for vectorized operations (faster than apply())
@pandas_udf(DoubleType())
def compound_growth(salary: pd.Series) -> pd.Series:
    rate = 0.05  # 5% annual growth
    return salary * np.power(1 + rate, 3)   # after 3 years

df.withColumn("salary_3yr", F.round(compound_growth(col("salary")), 2)).show()

## 5. Pandas UDF — Multiple Input Columns

In [0]:
from pyspark.sql.types import StringType

@pandas_udf(StringType())
def build_profile(name: pd.Series, department: pd.Series, salary: pd.Series) -> pd.Series:
    return name + " [" + department + "] $" + salary.astype(int).astype(str)

df.withColumn("profile", build_profile(col("name"), col("department"), col("salary"))).show(truncate=False)

## 6. applyInPandas — Grouped Transformation

Runs a Pandas function on each group of a grouped DataFrame.
Each group becomes a pandas DataFrame; the function returns a pandas DataFrame.
Use for: per-group normalization, per-group model scoring, complex reshaping.

In [0]:
# Normalize salary within each department (z-score normalization)
output_schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("salary_zscore", DoubleType(), True),
])

def normalize_salary(pdf: pd.DataFrame) -> pd.DataFrame:
    mean = pdf["salary"].mean()
    std  = pdf["salary"].std()
    pdf["salary_zscore"] = ((pdf["salary"] - mean) / std).round(4)
    return pdf[["emp_id", "name", "department", "salary", "salary_zscore"]]

df_normalized = df.groupby("department").applyInPandas(normalize_salary, schema=output_schema)
df_normalized.orderBy("department", "salary").show()

## 7. mapInPandas — Partition-Level Transformation

Unlike `applyInPandas`, `mapInPandas` operates on each **partition**
(not group) and doesn't require a shuffle. Good for enrichment from a
non-serializable model or external lookup.

In [0]:
def add_level(iterator):
    for pdf in iterator:
        pdf["level"] = pdf["salary"].apply(
            lambda s: "Senior" if s >= 95000 else "Mid" if s >= 75000 else "Junior"
        )
        yield pdf

output_schema2 = StructType(df.schema.fields + [StructField("level", StringType(), True)])

df_leveled = df.mapInPandas(add_level, schema=output_schema2)
df_leveled.show()

## 8. Performance Comparison Note

Run this conceptually — in real benchmarks the order is:

```
Built-in function    → Fastest  (Catalyst, JVM, no serialization)
Pandas UDF           → Fast     (Arrow columnar, batch processing)
Python UDF           → Slowest  (row-by-row, Python pickle serialization)
```

**Rule of thumb:**
1. Always check `pyspark.sql.functions` first — it likely has what you need.
2. Use Pandas UDF when you need numpy/scipy/sklearn logic.
3. Use Python UDF only when you have simple custom logic and don't care about performance.

In [0]:
# Equivalent operations — always prefer built-in
from pyspark.sql.functions import upper, when

# Bad: Python UDF for something built-in handles
bad_udf = udf(lambda s: s.upper() if s else None, StringType())
df.withColumn("name_upper_udf", bad_udf(col("name")))   # slow

# Good: use built-in
df.withColumn("name_upper",     upper(col("name"))).show()  # fast

## Key Takeaways

| UDF Type | Decorator / API | Input Type | Output Type | Speed |
|---|---|---|---|---|
| Python UDF | `@udf(returnType=T)` | Row scalar | Scalar | Slow |
| Python UDF | `udf(func, T)` | Row scalar | Scalar | Slow |
| Pandas Scalar | `@pandas_udf(T)` | `pd.Series` | `pd.Series` | Fast |
| Grouped Map | `.groupby().applyInPandas(f, schema)` | `pd.DataFrame` | `pd.DataFrame` | Medium |
| Partition Map | `.mapInPandas(f, schema)` | Iterator of `pd.DataFrame` | Iterator | Medium |

- Register UDFs for SQL: `spark.udf.register("name", func, type)`
- Handle None/null explicitly inside UDF body
- UDFs cannot be optimized by Catalyst — keep logic out of UDFs where possible